# Fine-tune a Cross-Encoder to Predict ROUGE-1

**The problem this fixes:** off-the-shelf BGE-reranker scores semantic relevance, which
disagreed catastrophically with ROUGE on this competition (your diagnostic showed
ROUGE=1.0 candidates being thrown away for semantically-better-but-lexically-different ones).

**The fix:** fine-tune a cross-encoder on regression. It predicts, given (question, candidate),
the **ROUGE-1 F1 score** the candidate would get against the unseen reference. Now
picking argmax is picking the candidate that maximizes the actual leaderboard metric.

**Pipeline:**
1. BGE-M3 retrieves top-K candidates for every training row (self-excluded).
2. ROUGE-1 of each candidate vs that row's reference = regression target.
3. Fine-tune BGE-reranker-v2-m3 with MSE loss on those (q, cand, rouge) triples.
4. Evaluate on val and compare against bi-encoder baseline.
5. Build the test submission only if val improves.

**Compute:** RTX 4050. Pair construction ~5-10 min. Training one epoch on ~300k pairs:
~30-50 min. Total under an hour.

**Honest caveat:** this is the right shape of fix but not guaranteed to work. The
model has to learn from question alone what makes a candidate ROUGE-close to a
reference it never sees. Train, evaluate on val, only commit if it actually beats
the bi-encoder top-1.

## 1 - Setup

In [1]:
!pip uninstall -y torchvision torchaudio
!pip install -q -U "sentence-transformers>=3.0.0" "transformers>=4.46.0,<5.0.0" \
    "rouge-score>=0.1.2" "sentencepiece>=0.2.0" "datasets>=3.0.0" "scikit-learn"

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timm 1.0.26 requires torchvision, which is not installed.
f

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import os, gc, json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device: cuda
gpu: Tesla T4 (15.6 GB)


## 2 - Load data + scorer

In [3]:
DATA_DIR = Path("/kaggle/input/datasets/offeibekoe/multilingual-health-challenge")
train = pd.read_csv(DATA_DIR/"Train.csv")
val   = pd.read_csv(DATA_DIR/"Val.csv")
test  = pd.read_csv(DATA_DIR/"Test.csv")

QCOL, ACOL, GCOL, IDCOL = "input", "output", "subset", "ID"
for df in (train, val, test):
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna("").astype(str).str.strip()
train = train[(train[QCOL]!="")&(train[ACOL]!="")].reset_index(drop=True)
val   = val[(val[QCOL]!="")&(val[ACOL]!="")].reset_index(drop=True)
print(f"train {len(train)}  val {len(val)}  test {len(test)}")

class WhitespaceTokenizer:
    def tokenize(self, t): return [] if t is None else str(t).strip().split()
_SCORER = rouge_scorer.RougeScorer(["rouge1"], tokenizer=WhitespaceTokenizer(), use_stemmer=False)
def rouge1(p, r): return _SCORER.score(str(r), str(p))["rouge1"].fmeasure

train 29814  val 6686  test 2618


## 3 - BGE-M3 retrieves top-K per row, with self-exclusion on train

Self-exclusion is critical: a training row must never retrieve its own answer,
or ROUGE target = 1.0 trivially and the model learns nothing useful.

In [4]:
from sentence_transformers import SentenceTransformer

class STEncoder:
    def __init__(self, name): self.model = SentenceTransformer(name, device=DEVICE)
    def encode(self, texts):
        return self.model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=False, batch_size=64,
                                 convert_to_numpy=True)

K = 10
bi = STEncoder("BAAI/bge-m3")

def encode_subset_indices(df, qcol, acol, gcol, encoder, k):
    indices = {}
    for g, grp in df.groupby(gcol):
        embs = encoder.encode(grp[qcol].tolist())
        nn = NearestNeighbors(n_neighbors=min(k+1, len(grp)), metric="cosine").fit(embs)
        indices[g] = {"nn": nn, "embs": embs,
                      "ans": np.array(grp[acol].astype(str).tolist(), dtype=object),
                      "orig_idx": np.array(grp.index.tolist())}
    return indices

t0 = time.time()
print("Encoding train and building per-subset indices...")
train_idx = encode_subset_indices(train, QCOL, ACOL, GCOL, bi, K)
print(f"  done in {time.time()-t0:.1f}s")

2026-06-03 00:23:21.366218: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780446201.513866      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780446201.560938      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780446201.940633      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780446201.940674      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780446201.940677      22 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Encoding train and building per-subset indices...
  done in 193.0s


## 4 - Build training pairs (q, candidate, rouge_target)

For every train row: retrieve top-K from same subset (skip self), label each
candidate with its ROUGE-1 against the reference.

In [5]:
def build_training_pairs(df, indices, qcol, acol, gcol, k):
    pairs_q, pairs_c, pairs_y = [], [], []
    for g, grp in df.groupby(gcol):
        m = indices[g]
        dist, idx = m["nn"].kneighbors(m["embs"], n_neighbors=min(k+1, len(m["ans"])))
        for local_i, (orig_idx, row) in enumerate(grp.iterrows()):
            picked = 0
            ref = str(row[acol])
            q   = str(row[qcol])
            for j in idx[local_i]:
                if m["orig_idx"][j] == orig_idx:
                    continue
                cand = str(m["ans"][j])
                pairs_q.append(q)
                pairs_c.append(cand)
                pairs_y.append(rouge1(cand, ref))
                picked += 1
                if picked == k:
                    break
    return pairs_q, pairs_c, np.array(pairs_y, dtype=np.float32)

t0 = time.time()
tr_q, tr_c, tr_y = build_training_pairs(train, train_idx, QCOL, ACOL, GCOL, K)
print(f"Built {len(tr_q)} pairs in {time.time()-t0:.1f}s")
print("Target ROUGE-1 distribution:")
print(pd.Series(tr_y).describe().round(4))
print(f"Targets >= 0.5: {int((tr_y>=0.5).sum())} ({(tr_y>=0.5).mean()*100:.1f}%)")
print(f"Targets >= 0.9: {int((tr_y>=0.9).sum())} ({(tr_y>=0.9).mean()*100:.1f}%)")

Built 298140 pairs in 36.3s
Target ROUGE-1 distribution:
count    298140.0000
mean          0.3450
std           0.3065
min           0.0000
25%           0.1429
50%           0.2400
75%           0.3803
max           1.0000
dtype: float64
Targets >= 0.5: 56681 (19.0%)
Targets >= 0.9: 44123 (14.8%)


In [6]:
import gc, torch

# Move models to CPU first — this forces GPU memory to actually free
for name in ["bi", "model", "trained_model", "trainer"]:
    if name in globals():
        obj = globals()[name]
        try:
            if hasattr(obj, "model"):
                obj.model.cpu()
            elif hasattr(obj, "cpu"):
                obj.cpu()
        except Exception:
            pass
        del globals()[name]

# These hold no GPU tensors but free RAM
for name in ["train_idx", "val_cands"]:
    if name in globals():
        del globals()[name]

# Multiple gc passes for circular refs
for _ in range(3):
    gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

GPU free: 15.44 GB / 15.64 GB


**Read those statistics carefully.** If most targets are near 0 (no candidate
remotely matches the reference), the model won't have signal to learn from.
If many are near 1 (lots of near-duplicates in train), it has clear examples.
You want meaningful spread - both low and high targets.

## 5 - Fine-tune cross-encoder on (q, cand) -> ROUGE-1 regression

sentence-transformers' CrossEncoder supports regression natively via num_labels=1.
We start from BGE-reranker-v2-m3 (its multilingual priors are useful starting weights)
and nudge it toward predicting ROUGE instead of semantic relevance.

In [7]:
import gc, torch

# Move models to CPU first so GPU memory actually frees, then delete
for name in ["model", "trainer", "loss", "trained_model"]:
    if name in globals():
        obj = globals()[name]
        try:
            if hasattr(obj, "model") and hasattr(obj.model, "cpu"):
                obj.model.cpu()
            elif hasattr(obj, "cpu"):
                obj.cpu()
        except Exception:
            pass
        del globals()[name]

# train_ds is CPU only — drop it to free RAM (re-create in the training cell)
if "train_ds" in globals():
    del globals()["train_ds"]

for _ in range(3):
    gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

GPU free: 15.44 GB / 15.64 GB


In [8]:
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import MSELoss
from datasets import Dataset

train_ds = Dataset.from_dict({
    "query": tr_q,
    "candidate": tr_c,
    "label": tr_y.tolist(),
})
print(f"Training dataset: {len(train_ds)} pairs")

OUT_DIR = "rouge_reranker_bgem3"
model = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    num_labels=1,
    max_length=512,
    device=DEVICE,
)
loss = MSELoss(model=model)

args = CrossEncoderTrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=1,
    report_to=["none"],
    dataloader_num_workers=2,
    seed=42,
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=loss,
)

t0 = time.time()
trainer.train()
print(f"Trained in {(time.time()-t0)/60:.1f} min")
model.save_pretrained(OUT_DIR)
print(f"Saved to {OUT_DIR}/")

Training dataset: 298140 pairs


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Step,Training Loss
200,6.704700
400,0.083500
600,0.070700
800,0.064600
1000,0.062700
1200,0.059500
1400,0.056400
1600,0.055400
1800,0.056100
2000,0.055000


Trained in 317.6 min
Saved to rouge_reranker_bgem3/


## 6 - Evaluate on val: does it actually beat the bi-encoder baseline?

Three numbers per subset:
- bi_top1: BGE-M3's choice (current best)
- rouge_rerank: the new trained reranker's choice
- oracle_topK: the ceiling

In [9]:
def retrieve_topk_for(df, indices, qcol, gcol, encoder, k):
    cands = [[] for _ in range(len(df))]
    pos = {idx:i for i,idx in enumerate(df.index)}
    for g, grp in df.groupby(gcol):
        m = indices.get(g) or next(iter(indices.values()))
        embs = encoder.encode(grp[qcol].tolist())
        k_eff = min(k, len(m["ans"]))
        _, idx = m["nn"].kneighbors(embs, n_neighbors=k_eff)
        for row_idx, irow in zip(grp.index, idx):
            cands[pos[row_idx]] = [str(m["ans"][j]) for j in irow]
    return cands

val_cands = retrieve_topk_for(val, train_idx, QCOL, GCOL, bi, K)
print(f"Retrieved top-{K} for {len(val_cands)} val rows")

trained_model = CrossEncoder(OUT_DIR, num_labels=1, max_length=512, device=DEVICE)
flat_pairs, row_lens = [], []
for q, cs in zip(val[QCOL].tolist(), val_cands):
    flat_pairs.extend([(q, c) for c in cs])
    row_lens.append(len(cs))
print(f"Scoring {len(flat_pairs)} pairs on the fine-tuned reranker...")
t0 = time.time()
scores = trained_model.predict(flat_pairs, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
print(f"  done in {time.time()-t0:.1f}s")

rerank_top1, off = [], 0
for cs, n in zip(val_cands, row_lens):
    if n == 0: rerank_top1.append(""); continue
    row_scores = scores[off:off+n]; off += n
    rerank_top1.append(cs[int(np.argmax(row_scores))])

def per_subset_rouge(preds, refs, subs):
    sub = np.array(subs); rows = []
    for s in np.unique(sub):
        m = sub == s
        rows.append({"subset":s, "n":int(m.sum()),
                     "r1": round(float(np.mean([rouge1(p, r) for p, r, k in zip(preds, refs, m) if k])), 4)})
    rows.append({"subset":"OVERALL(micro)", "n":len(preds),
                 "r1": round(float(np.mean([rouge1(p, r) for p, r in zip(preds, refs)])), 4)})
    return pd.DataFrame(rows)

bi_top1 = [c[0] if c else "" for c in val_cands]
oracle  = [max(cs, key=lambda c: rouge1(c, r)) if cs else "" for cs, r in zip(val_cands, val[ACOL])]

report = (per_subset_rouge(bi_top1, val[ACOL].tolist(), val[GCOL].tolist())
          .rename(columns={"r1":"bi_top1"})
          .merge(per_subset_rouge(rerank_top1, val[ACOL].tolist(), val[GCOL].tolist())
                 .rename(columns={"r1":"rouge_rerank"}), on=["subset","n"])
          .merge(per_subset_rouge(oracle, val[ACOL].tolist(), val[GCOL].tolist())
                 .rename(columns={"r1":f"oracle_top{K}"}), on=["subset","n"]))
report["delta"] = (report["rouge_rerank"] - report["bi_top1"]).round(4)
print(report.to_string(index=False))

NameError: name 'train_idx' is not defined

**Read the `delta` column.** Positive = trained reranker beats bi-encoder.
Negative = the fine-tune didn't work for that subset.

If overall delta is positive: build the submission below. If negative: don't
submit this — the BGE-M3-only baseline stays your best. The training may need
more epochs, more data, or a different backbone — but only commit when the
numbers say it works.

## 7 - Build test submission (only run if val showed positive delta)

Test corpus = train + val (val is fair game for test-time retrieval).

In [ ]:
corpus = pd.concat([train, val], ignore_index=True).reset_index(drop=True)
print(f"Corpus: {len(corpus)} (train+val)")

corpus_idx = encode_subset_indices(corpus, QCOL, ACOL, GCOL, bi, K)
test_cands = retrieve_topk_for(test, corpus_idx, QCOL, GCOL, bi, K)

flat_pairs, row_lens = [], []
for q, cs in zip(test[QCOL].tolist(), test_cands):
    flat_pairs.extend([(q, c) for c in cs])
    row_lens.append(len(cs))
print(f"Scoring {len(flat_pairs)} test pairs...")
t0 = time.time()
scores = trained_model.predict(flat_pairs, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
print(f"  done in {time.time()-t0:.1f}s")

test_pred, off = [], 0
for cs, n in zip(test_cands, row_lens):
    if n == 0: test_pred.append(""); continue
    row_scores = scores[off:off+n]; off += n
    test_pred.append(cs[int(np.argmax(row_scores))])

import re
clean = [re.sub(r"<extra_id_\d+>", "", str(p)).strip() for p in test_pred]
sub = pd.DataFrame({"ID":test[IDCOL], "TargetRLF1":clean, "TargetR1F1":clean, "TargetLLM":clean})
sub.to_csv("submission_rouge_reranker.csv", index=False, encoding="utf-8")
print(f"Saved submission_rouge_reranker.csv ({len(sub)} rows)")